# LLM with Key Value Caching


In [1]:
import os

# Sets the high watermark ratio to 0.0, completely disabling the upper memory allocation limits
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"

In [2]:
checkpoint_dir = "checkpoints/"

In [3]:
import gc
import torch


def clean_memory_cache():

    if torch.mps.is_available():
        print(
            f"Before Clearing, Available memory: {torch.mps.driver_allocated_memory() / 1024 / 1024:.2f} MB"
        )
        gc.collect()
        torch.mps.empty_cache()
    elif torch.cuda.is_available():
        print(
            f"Before Clearing, Available memory: {torch.cuda.memory_allocated() / 1024 / 1024:.2f} MB"
        )
        gc.collect()
        torch.cuda.empty_cache()
    else:
        gc.collect()
        return

## Transformer and Vision Transformer

In [4]:
from typing import Optional, Tuple, Any, List
import torch
import torch.nn as nn
from dataclasses import dataclass

import numpy as np
import torch
import torchvision
from torch import nn, optim
from torchvision import transforms
import sklearn
from sklearn.metrics import confusion_matrix
import tqdm
import copy
from torch.utils.data import DataLoader, Subset

torch.autograd.set_detect_anomaly(True)
device = (
    torch.accelerator.current_accelerator().type
    if torch.accelerator.is_available()
    else "cpu"
)
import os

num_workers = min(2, os.cpu_count())
import logging

# Configure logging to write directly to a file
logging.basicConfig(
    filename="vit.log",
    filemode="a",  # 'a' appends to the file, 'w' overwrites it on every run
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)


@dataclass(init=True)
class TrainResult:
    """
    A collection containing everything we need to know about the training results
    """

    train_losses: List[float]
    train_accs: List[float]
    val_accs: List[float]
    val_losses: List[float]


result = TrainResult(train_losses=[], train_accs=[], val_accs=[], val_losses=[])


class AverageMeter:
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0.0
        self.avg = 0.0
        self.sum = 0.0
        self.count = 0.0

    def update(self, val: float, n: int = 1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count if self.count > 0 else 0.0

    def calculate(self) -> float:
        return self.avg


def print_variance(name: str, data: torch.Tensor):
    # Compute variance across features/neurons and average across the batch
    neuron_variance = torch.mean(torch.var(data.detach().float(), dim=-1))
    print(f"{name}: Variance = {neuron_variance.item():.6f}")


class MultiHeadedAttention(nn.Module):
    def __init__(self, dim: int, n_hidden: int, num_heads: int):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.n_hidden = n_hidden
        self.scale = n_hidden**-0.5

        self.qkv_projection = nn.Linear(dim, num_heads * n_hidden * 3, bias=False)
        self.W0 = nn.Linear(num_heads * n_hidden, dim)

    def forward(
        self, x: torch.Tensor, attn_mask: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        B, T, _ = x.shape

        # Shape: (B, T, 3 * num_heads * n_hidden) -> (B, num_heads, T, 3 * n_hidden)
        qkv = (
            self.qkv_projection(x)
            .reshape(B, T, self.num_heads, 3 * self.n_hidden)
            .transpose(1, 2)
        )
        q, k, v = qkv.chunk(3, dim=-1)

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale

        if attn_mask is not None:
            if attn_mask.dim() == 3:
                attn_mask = attn_mask.unsqueeze(1)
            scores = scores.masked_fill(attn_mask == 0, float("-inf"))

        attn_weights = torch.softmax(scores, dim=-1)

        context = torch.matmul(attn_weights, v)
        context = context.transpose(1, 2).reshape(B, T, self.num_heads * self.n_hidden)

        output = self.W0(context)
        return output, attn_weights


class AttentionResidual(nn.Module):
    def __init__(self, dim: int, attn_dim: int, mlp_dim: int, num_heads: int):
        super().__init__()
        self.attn = MultiHeadedAttention(dim, attn_dim, num_heads)

        # LayerNorm applied inside the FFN sequence only
        self.ffn = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, dim),
        )

    def forward(
        self, x: torch.Tensor, attn_mask: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        # Attention block with residual connection (no norm)
        attn_out, alphas = self.attn(x, attn_mask=attn_mask)
        x = x + attn_out

        # FFN block with residual connection (norm is first layer inside self.ffn)
        x = x + self.ffn(x)
        return x, alphas


class Transformer(nn.Module):
    def __init__(
        self, dim: int, attn_dim: int, mlp_dim: int, num_heads: int, num_layers: int
    ):
        super().__init__()
        self.layers = nn.ModuleList(
            [
                AttentionResidual(dim, attn_dim, mlp_dim, num_heads)
                for _ in range(num_layers)
            ]
        )

    def forward(
        self,
        x: torch.Tensor,
        attn_mask: Optional[torch.Tensor] = None,
        return_attn: bool = False,
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        collected_attns = []

        for layer in self.layers:
            x, alphas = layer(x, attn_mask=attn_mask)
            if return_attn:
                collected_attns.append(alphas)

        if return_attn:
            return x, torch.stack(collected_attns, dim=1)
        return x, None


class PatchEmbed(nn.Module):
    def __init__(self, img_size: int, patch_size: int, nin: int, nout: int):
        super().__init__()
        assert (
            img_size % patch_size == 0
        ), "Image dimensions must be divisible by patch size."

        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2

        self.proj = nn.Conv2d(
            in_channels=nin,
            out_channels=nout,
            kernel_size=patch_size,
            stride=patch_size,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Output: (B, num_patches, nout) in float
        return self.proj(x).flatten(2).transpose(1, 2)


class VisionTransformer(nn.Module):
    def __init__(
        self,
        n_channels: int,
        nout: int,
        img_size: int,
        patch_size: int,
        dim: int,
        attn_dim: int,
        mlp_dim: int,
        num_heads: int,
        num_layers: int,
        num_global_tokens: int = 1,  # Number of global/CLS tokens
    ):
        super().__init__()
        self.num_global_tokens = num_global_tokens

        self.patch_embed = PatchEmbed(
            img_size=img_size, patch_size=patch_size, nin=n_channels, nout=dim
        )
        num_patches = self.patch_embed.num_patches

        # Learnable global/CLS tokens of shape (1, num_global_tokens, dim)

        self.cls_tokens = nn.Parameter(torch.zeros(1, num_global_tokens, dim))

        # Position embeddings covering both global tokens and image patches
        total_seq_len = num_global_tokens + num_patches
        self.pos_embed = nn.Parameter(torch.zeros(1, total_seq_len, dim))

        nn.init.trunc_normal_(self.cls_tokens, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        self.transformer = Transformer(
            dim=dim,
            attn_dim=attn_dim,
            mlp_dim=mlp_dim,
            num_heads=num_heads,
            num_layers=num_layers,
        )

        # Classification / Projection Head
        self.head = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, nout),
        )

    def forward(
        self, img: torch.Tensor, return_attn: bool = False
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        B = img.shape[0]

        # 1. Patch embeddings: (B, num_patches, dim)
        embs = self.patch_embed(img)

        # 2. Expand global tokens across batch: (B, num_global_tokens, dim)
        cls_tokens = self.cls_tokens.expand(B, -1, -1)

        # 3. Concatenate global tokens with patch embeddings: (B, num_global_tokens + num_patches, dim)
        x = torch.cat((cls_tokens, embs), dim=1)

        # 4. Add position embeddings
        x = x + self.pos_embed

        # 5. Transformer forward pass
        x, alphas = self.transformer(x, attn_mask=None, return_attn=return_attn)

        # 6. Extract representations of all global tokens: (B, num_global_tokens, dim)
        global_repr = x[:, : self.num_global_tokens]

        # Aggregate global tokens:
        # - Option A: Use the primary token (index 0) if acting as a single CLS token with register tokens
        # out = self.head(global_repr[:, 0])
        #
        # - Option B: Mean-pool across all global tokens
        out = self.head(global_repr.mean(dim=1))  # (B, nout)

        return out, alphas


# evaluate the model
@torch.compile
def evaluate_cifar_model(model, criterion, val_loader):
    is_train = model.training
    model.eval()
    with torch.no_grad():
        loss_meter, acc_meter = AverageMeter(), AverageMeter()
        for img, labels in val_loader:
            # move all img, labels to device (cuda)
            img = img.to(device)
            labels = labels.to(device)
            outputs, _ = model(img)
            loss_meter.update(criterion(outputs, labels).item(), len(img))
            acc = (outputs.argmax(-1) == labels).float().mean().item()
            acc_meter.update(acc, len(img))
    model.train(is_train)
    return loss_meter.calculate(), acc_meter.calculate()


result = TrainResult(train_losses=[], train_accs=[], val_losses=[], val_accs=[])


def main():
    model = VisionTransformer(
        n_channels=3,
        nout=10,
        img_size=32,
        patch_size=4,
        dim=128,
        attn_dim=64,
        mlp_dim=128,
        num_heads=3,
        num_layers=6,
    ).to(device)
    print(model)
    MEAN = [0.4914, 0.4822, 0.4465]
    STD = [0.247, 0.2435, 0.2616]
    img_transform = transforms.Compose(
        [transforms.ToTensor(), transforms.Normalize(mean=MEAN, std=STD)]
    )
    inv_transform = transforms.Compose(
        [
            transforms.Normalize(mean=[0.0, 0.0, 0.0], std=1 / np.array(STD)),
            transforms.Normalize(mean=-np.array(MEAN), std=[1.0, 1.0, 1.0]),
            transforms.ToPILImage(),
        ]
    )
    train_dataset = torchvision.datasets.CIFAR10(
        train=True, root="data", transform=img_transform, download=True
    )
    val_dataset = torchvision.datasets.CIFAR10(
        train=False, root="data", transform=img_transform
    )
    train_dataloader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=256,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
    )
    val_dataloader = torch.utils.data.DataLoader(
        val_dataset,
        batch_size=256,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )
    criterion = nn.CrossEntropyLoss()
    NUM_EPOCHS = 10
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.003)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=NUM_EPOCHS, eta_min=1e-05
    )
    # result = TrainResult(train_losses=[], train_accs=[], val_losses=[], val_accs=[])
    for epoch in range(NUM_EPOCHS):
        loss_meter = AverageMeter()
        acc_meter = AverageMeter()
        for img, labels in tqdm.tqdm(
            train_dataloader, desc="Training at " + str(epoch)
        ):
            img, labels = (img.to(device), labels.to(device))
            optimizer.zero_grad()
            outputs, _ = model(img)
            loss = criterion(outputs, labels)
            loss_meter.update(loss.item(), len(img))
            acc = (outputs.argmax(-1) == labels).float().mean().item()
            acc_meter.update(acc, len(img))
            loss.backward()
            optimizer.step()
        scheduler.step()
        result.train_losses.append(loss_meter.calculate())
        result.train_accs.append(acc_meter.calculate())
        print(
            f"Train Epoch: {epoch}, Loss: {loss_meter.calculate()}, Acc: {acc_meter.calculate()}"
        )
        val_loss, val_acc = evaluate_cifar_model(model, criterion, val_dataloader)
        result.val_losses.append(val_loss)
        result.val_accs.append(val_acc)
        print(f"Val Epoch: {epoch}, Loss: {val_loss}, Acc: {val_acc}")
        best_models = "Vision_transformer_Best_" + str(epoch + 1) + ".pt"
        best_save_path = os.path.join(checkpoint_dir, best_models)
        model_path = "Vision_transformer_" + str(epoch + 1) + ".pt"
        save_path = os.path.join(checkpoint_dir, model_path)

        # todo add an early stopping criteria and restore best weights

        if epoch > 0 and result.val_losses[epoch] > result.val_losses[epoch - 1]:
            torch.save(model.state_dict(), best_save_path)
            val_loss, val_acc = evaluate_cifar_model(model, criterion, val_dataloader)
            print(f"Val Epoch: {epoch + 1}, Loss: {val_loss}, Acc: {val_acc}")
            print("Finished Training")

        else:
            torch.save(model.state_dict(), save_path)
        val_loss, val_acc = evaluate_cifar_model(model, criterion, val_dataloader)
        clean_memory_cache()
        print(f"Val Epoch: {epoch + 1}, Loss: {val_loss}, Acc: {val_acc}")
    print("Finished Training")

In [5]:
# import os
#
!pip install wget
import wget

#
if not os.path.exists("shakespeare.txt"):
    wget.download(
        "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    )

In [6]:
file = "input.txt"
with open(file, "r") as f:
    dialogues = f.read()

In [7]:
all_dialogues = dialogues.split("\n\n")

In [8]:
for line in all_dialogues[:10]:
    print(line)

First Citizen:
Before we proceed any further, hear me speak.
All:
Speak, speak.
First Citizen:
You are all resolved rather to die than to famish?
All:
Resolved. resolved.
First Citizen:
First, you know Caius Marcius is chief enemy to the people.
All:
We know't, we know't.
First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?
All:
No more talking on't; let it be done: away, away!
Second Citizen:
One word, good citizens.
First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.


In [9]:
import nltk

nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [10]:
def tokenize(s):
    return nltk.word_tokenize(s)

## Dialogue LLM

In [11]:
def tokenize(s):
    return nltk.word_tokenize(s)


class MyTokenizer:
    def __init__(self, raw_text: str):
        # raw_text     contains the text from which we will build our vocabulary

        self.start = "<START>"  # token that starts every example
        self.pad = "<PAD>"  # token used to pad examples to the same length
        self.unk = "<UNK>"  # token used if encountering a word not in our vocabulary

        vocab = np.unique(tokenize(raw_text))
        vocab = np.concatenate([np.array([self.start, self.pad, self.unk]), vocab])

        self.vocab = vocab  # array of tokens in order
        self.tok_to_id = {w: i for i, w in enumerate(vocab)}  # mapping of token to ID
        self.id_to_token = {i: w for i, w in enumerate(vocab)}
        self.vocab_size = len(self.vocab)  # size of vocabulary

    def __len__(self):
        return self.vocab_size

    def encode(self, s: str) -> torch.Tensor:
        # s           input string
        #
        # Output
        # id_tensor   a tensor of token ids, starting with the start token.t

        id_tensor = torch.from_numpy(
            np.array(
                [self.tok_to_id[self.start]]
                + [self.tok_to_id[w] for w in tokenize(s) if w in self.tok_to_id],
                dtype=np.int32,
            )
        )

        # TODO: tokenize the input using word_tokenize. Return a tensor  of the token ids, starting with the token id for the start token.
        # ============ ANSWER START ===========
        # encoded_string = tokenize(s)
        # token_ids =
        # token_ids.append(self.tok_to_id[self.start])
        # token_ids.extend(
        #     [self.tok_to_id[w] for w in encoded_string if w in self.tok_to_id.keys()]
        # )
        # id_tensor = np.array(token_ids)

        # id_tensor = torch.from_numpy(id_tensor)
        # ============ ANSWER END =============

        return id_tensor

    def decode(self, toks: torch.Tensor) -> str:
        # toks         a list of token ids
        #
        # Output
        # decoded_str  the token ids decoded back into a string (join with a space)

        # TODO: convert the token ids back to the actual corresponding words.
        # Join the tokens with a space and return the full string
        # ============ ANSWER START ===========
        return " ".join(
            [
                self.id_to_token[int(token)]
                for token in toks
                if token in self.tok_to_id.values()
            ]
        ).rstrip()

        # ============ ANSWER END =============

        # return decoded_str

    def pad_examples(self, tok_list: List[torch.Tensor]) -> torch.Tensor:
        # Pads the tensors to the right with the pad token so that they are the same length.
        #
        # tok_list       a list of tensors containing token ids (maybe of different lengths)
        #
        # Output
        # padded_tokens  shape: (len(tok_list), max length within tok_list)
        return torch.nn.utils.rnn.pad_sequence(
            tok_list, batch_first=True, padding_value=self.tok_to_id[self.pad]
        )


tok = MyTokenizer(dialogues)

In [12]:
len(tok)

14213

In [13]:
# tokenizer test cases
input_string = "KING RICHARD III:\nSay that I did all this for love of her. bluye"
enc = tok.encode(input_string)
print(enc)

# for x in enc:
#     print(x, type(x), x in tok.tok_to_id.values(), x in tok.id_to_token.values())
dec = tok.decode(enc)
print(dec)
# print("<START> KING RICHARD III : Say that I did all this for love of her .")
assert dec == "<START> KING RICHARD III : Say that I did all this for love of her ."

tensor([    0,  1593,  2182,  1481,   223,  2343, 12742,  1476,  5704,  3319,
        12795,  6848,  8727,  9608,  7655,   221], dtype=torch.int32)
<START> KING RICHARD III : Say that I did all this for love of her .


In [14]:
class DialogueDataset:
    def __init__(self, tokenizer: MyTokenizer, lines: List[str], max_N: int):
        # tokenizer    an instance of MyTokenizer
        # lines        a list of strings. each element in an example in the dataset
        # max_N        the maximum number of tokens allowed per example. More than this will be truncated
        self.lines = lines
        self.tokenizer = tokenizer
        self.max_N = max_N

    def __len__(self) -> int:
        return len(self.lines)

    # def __iter__(self):
    #     for line in self.lines:
    #         yield self.tokenizer.encode(line)[: self.max_N]

    def __getitem__(self, idx: int) -> torch.Tensor:
        # returns the example at int encoded by the tokenizer
        # truncates the example if it is more than max_N tokens
        return self.tokenizer.encode(self.lines[idx])[: self.max_N]

    # def __getitems__(self,indices:int):
    #     return [self.__getitem__(idx) for idx in indices]

In [15]:
ds = DialogueDataset(tok, all_dialogues, max_N=200)

In [16]:
def collate_fn(examples: List[torch.Tensor]):
    """
    # examples        a batch of tensors containing token ids (maybe of different lengths)
    # Outputs a dictionary containing
    #   input_ids     a single tensor with all of the examples padded (from the right) to the max
    #                 length within the batch. shape:(B, max length within examples)
    #   input_mask    a tensor indicating which tokens are padding and should be ignored. 0 if padding
    #                 and 1 if not. shape: (B, max length within examples)
    """
    new_input_ids = tok.pad_examples(examples)
    attn_mask = torch.ones(new_input_ids.shape)  # 1s should not be ignored

    # causal attention mask
    attn_mask[new_input_ids == tok.tok_to_id[tok.pad]] = (
        0  # should be ignored if it is a padded
    )
    return {"input_ids": new_input_ids, "input_mask": attn_mask}

In [17]:
ds[1]

tensor([    0,   323,   223,  2517,   219, 12008,   221], dtype=torch.int32)

In [18]:
tokens = [ds[1], ds[2]]

In [19]:
type(tokens)

list

In [20]:
collate_fn(tokens)

{'input_ids': tensor([[    0,   323,   223,  2517,   219, 12008,   221,     1,     1,     1,
              1,     1,     1,     1,     1],
         [    0,  1151,   708,   223,  3050,  3506,  3319, 10983, 10724, 12921,
           5706, 12733, 12921,  6507,   225]], dtype=torch.int32),
 'input_mask': tensor([[1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
         [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]])}

In [21]:
tokens[0]

tensor([    0,   323,   223,  2517,   219, 12008,   221], dtype=torch.int32)

In [22]:
tokens[1]

tensor([    0,  1151,   708,   223,  3050,  3506,  3319, 10983, 10724, 12921,
         5706, 12733, 12921,  6507,   225], dtype=torch.int32)

In [23]:
from torch.utils.data import DataLoader, Subset

In [24]:
tok

In [25]:
train_dl = DataLoader(ds, batch_size=16, num_workers=0, collate_fn=collate_fn)

In [26]:
BATCH_SIZE = 16
BUFFER_SIZE = 4

In [27]:
class DialogueGPT(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        max_N: int,
        dim: int,
        attn_dim: int,
        mlp_dim: int,
        num_heads: int,
        num_layers: int,
    ):
        # vocab_size       size of the vocabulary
        # max_N            maximum number of tokens allowed to appear in 1 example
        # dim              embedding dimension
        # attn_dim         the hidden dimension of the attention layer
        # mlp_dim          the hidden layer dimension of the FFN
        # num_heads        the number of heads in the attention layer
        # num_layers       the number of attention layers.

        super().__init__()
        """
        • Given the token ids, retrieve the corresponding token embeddings. Add to this a learned positional embedding.
        • Generate a causal attention mask. Remember that for GPT, every token only depends on itself and the tokens before it
        • Pass the embeddings and the attention mask to the transformer and the language model head. Output logits of size (T ×V) where T is the number of tokens and V is the vocabulary size. This step is implemented for you
        """
        # TODO: set up the token embedding and positional embeddings
        #       Hint, use nn.Embedding
        # Already Padded to keep sequence length same
        # Next use Graph Neural Network
        # token_embeddings

        self.token_embeddings = nn.Embedding(
            num_embeddings=vocab_size, embedding_dim=dim
        )

        self.pos_embeddings = nn.Embedding(num_embeddings=max_N, embedding_dim=dim)

        self.transformer = Transformer(
            dim=dim,
            attn_dim=attn_dim,
            mlp_dim=mlp_dim,
            num_heads=num_heads,
            num_layers=num_layers,
        )
        # Projection Head
        self.head = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, vocab_size))

    def forward(
        self, input_ids: torch.Tensor, return_attn=False
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        # input_ids     a batch of input ids (right padded). shape: (B x T)
        # return_attn   whether to return the attention weights
        #
        # Output
        # out           the logit vector (B x T x V)
        # alphas        the attention weights if return_attn is True. Otherwise None shape: (B, num_layers, num_heads, T, T)
        """

        Args:
            input_ids: Batch of input ids right padded shape: (BxT)
            return_attn: whether to return the attention weights
        • Generate a causal attention mask. Remember that for Generative  Pretrained Transformer, every token only depends on itself and the tokens before it
        • Pass the embeddings and the attention mask to the transformer and the language model head. Output logits of size (T ×V) where T is the number of tokens and V is the vocabulary size. This step is implemented for you
        Returns:

        """
        B, T = input_ids.shape
        pos_ids = torch.arange(0, T, dtype=torch.long, device=device).unsqueeze(
            0
        )  # Shape: (1, T)
        embs = self.token_embeddings(input_ids) + self.pos_embeddings(pos_ids)
        # TODO: retrieve the token embeddings for the input_ids.
        #       Add to the token embeddings the positional embeddings.
        #       Store the combined embedding in embs

        # TODO: Create the causal attention mask, which should be of size (B, T, T)
        #       Remember that the causal attention mask is lower triangular (all tokens only
        #       depend on themselves and the tokens before them).   Store the mask in causal_attn_mask
        # Hint: check out torch.tril creates a causal mask where each token can only attend to previous tokens and itself
        causal_attn_mask = (
            torch.tril(torch.ones(T, T)).unsqueeze(0).repeat(B, 1, 1)
        ).to(
            device
        )  # Shape: (B, T, T)
        # ============ ANSWER START ============

        # ============ ANSWER END ==============

        x, alphas = self.transformer(
            embs, attn_mask=causal_attn_mask, return_attn=return_attn
        )
        out = self.head(x)
        return out, alphas

    def generate(self, input_ids, num_tokens):
        # you can assume batch size 1
        # greedy generation
        with torch.no_grad():
            for i in range(num_tokens):
                out, _ = self.forward(input_ids)
                new_token = torch.argmax(out[:, [-1]], -1)
                input_ids = torch.cat([input_ids, new_token], dim=1)
        return input_ids

In [28]:
class DialogueLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.criterion = nn.CrossEntropyLoss(reduction="none")

    def forward(
        self, logits: torch.Tensor, input_ids: torch.Tensor, inp_mask: torch.Tensor
    ):
        """
        # logits      the logits produced by DialogueGPT. shape: (B x T x V)
        # input_ids   the token ids. shape: (B x T)
        # inp_mask    a 0/1 mask of which tokens are padding tokens and should be ignored. shape: (B x T)

        TODO: Implement the language model loss. For logits[i], we want to supervise the i+1 token_id with the cross entropy loss. We thus will not supervise the start token (input_ids[0]) or use the last logit vector (logits[-1]). Return the average of the losses for each token in the batch, making sure to ignore tokens corresponding to the padding (use inp_mask).
        """
        loss = 0

        # start_token = input_ids[0]
        relevant_logits = logits[:, :-1, :]
        # print(f"{logits.shape=},{relevant_logits.shape=}")
        relevant_tokens = input_ids[:, 1:]
        shift_mask = inp_mask[:, 1:]
        relevant_logits = relevant_logits.permute(0, 2, 1)
        # print(f"{relevant_logits.shape=}")

        loss = self.criterion(relevant_logits, relevant_tokens)
        shift_mask = shift_mask.to(dtype=loss.dtype, device=loss.device)
        loss *= shift_mask

        # ============ ANSWER START ============

        # ============ ANSWER END ==============
        return torch.sum(loss) / torch.clamp(torch.sum(shift_mask), min=1e-9)

In [29]:
import torch.optim as optim

model = DialogueGPT(
    vocab_size=tok.vocab_size,
    max_N=200,
    dim=128,
    attn_dim=64,
    mlp_dim=128,
    num_heads=3,
    num_layers=6,
).to(device)
criterion = DialogueLoss()

NUM_EPOCHS = 80

optimizer = optim.AdamW(
    model.parameters(), lr=0.0001, weight_decay=0
)  # implement in homework
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

In [30]:
model

DialogueGPT(
  (token_embeddings): Embedding(14213, 128)
  (pos_embeddings): Embedding(200, 128)
  (transformer): Transformer(
    (layers): ModuleList(
      (0-5): 6 x AttentionResidual(
        (attn): MultiHeadedAttention(
          (qkv_projection): Linear(in_features=128, out_features=576, bias=False)
          (W0): Linear(in_features=192, out_features=128, bias=True)
        )
        (ffn): Sequential(
          (0): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (1): Linear(in_features=128, out_features=128, bias=True)
          (2): GELU(approximate='none')
          (3): Linear(in_features=128, out_features=128, bias=True)
        )
      )
    )
  )
  (head): Sequential(
    (0): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=128, out_features=14213, bias=True)
  )
)

In [ ]:
result = TrainResult(train_losses=[], train_accs=[], val_accs=[], val_losses=[])

In [ ]:
# Ensure optimizer starts completely clean before the epoch loop begins
optimizer.zero_grad()

for epoch in range(NUM_EPOCHS):
    loss_meter = AverageMeter()

    # Wrap your loader securely
    for step, inp_dict in tqdm.tqdm(
        enumerate(train_dl), desc=f"Training at {epoch}", total=len(train_dl)
    ):
        inp_ids, inp_mask = inp_dict["input_ids"], inp_dict["input_mask"]

        inp_ids = inp_ids.to(device).long()
        inp_mask = inp_mask.to(device)

        # 1. Forward Pass
        outputs, _ = model(input_ids=inp_ids)

        # 2. FIX: Scale the loss down by BUFFER_SIZE to normalize gradients
        loss = criterion(outputs, inp_ids, inp_mask)
        scaled_loss = loss / BUFFER_SIZE

        # 3. Backward Pass (Accumulates gradients safely)
        scaled_loss.backward()

        # Track the true unscaled loss in your meter
        loss_meter.update(loss.item(), len(inp_dict["input_ids"]))

        # 4. FIX: Step the optimizer on buffer limit OR at the final step of the dataset
        if (step + 1) % BUFFER_SIZE == 0 or (step + 1) == len(train_dl):
            optimizer.step()
            optimizer.zero_grad()  # Resets the buffer for the next accumulation block

            # OPTIONAL: If your scheduler decays per-step instead of per-epoch,
            # place `scheduler.step()` right here.

    # 5. Step scheduler at the epoch level (if using an epoch-based scheduler)
    scheduler.step()

    # 6. Safe Text Generation Example (Switched to eval/inference mode to protect memory buffers)
    model.eval()
    with torch.inference_mode():
        # Ensure your start prompt token matches your vocabulary bounds
        inp = tok.encode("").unsqueeze(0).to(device)
        generated_output = model.generate(inp, 10)
        print(f"\n[Generated Sample]: {tok.decode(generated_output[0].cpu())}")

    model.train()  # Switch back to training mode for the next epoch loop
    clean_memory_cache()
    result.train_losses.append(loss_meter.calculate())

    print(
        f"Train Epoch: {epoch}, Loss: {loss_meter.calculate():0.4f}, LR: {scheduler.get_last_lr()[0]}"
    )

Training at 0: 100%|██████████| 452/452 [01:57<00:00,  3.85it/s]



[Generated Sample]: <START> : : , , , , , , , ,
Before Clearing, Available memory: 93.78 MB
Train Epoch: 0, Loss: 8.4712, LR: 9.996145181203615e-05


Training at 1: 100%|██████████| 452/452 [01:54<00:00,  3.94it/s]



[Generated Sample]: <START> : : , , , , , , , ,
Before Clearing, Available memory: 93.78 MB
Train Epoch: 1, Loss: 7.0322, LR: 9.98458666866564e-05


Training at 2: 100%|██████████| 452/452 [01:53<00:00,  3.99it/s]


...

Training at 35: 100%|██████████| 452/452 [01:51<00:00,  4.07it/s]



[Generated Sample]: <START> KING EDWARD IV : I will not be a man
Before Clearing, Available memory: 93.78 MB
Train Epoch: 35, Loss: 5.1138, LR: 5.782172325201153e-05


Training at 36: 100%|██████████| 452/452 [01:51<00:00,  4.04it/s]



[Generated Sample]: <START> KING EDWARD IV : I will not be a man
Before Clearing, Available memory: 93.78 MB
Train Epoch: 36, Loss: 5.0993, LR: 5.587686987289187e-05


Training at 37:  10%|▉         | 44/452 [00:10<01:30,  4.49it/s]

In [ ]:
inp = tok.encode("").unsqueeze(0).to(device)
print(tok.decode(model.generate(inp, 50)[0].cpu()))